# 03 · Lake elevation and volume change

For one lake polygon, from the per-tile zarr stacks (`tile_zarrs`):

1. pick the tile's stack and index its strips;
2. **time series** of the lake surface relative to a background ring around it;
3. **ice thickness** under the lake (BedMachine) and the lake's width-to-thickness ratio d/H;
4. *(optional, slower)* **outline and events**: where the surface changes (SD outline) and the drain/fill events between observations, with their volumes.

Results go to `OUTPUT_DIR/volume/notebook_<lake>_<date>/`. All the functions are in `lake_volume.py` (repo root); `03_volume_batch.py` runs the same steps for every lake.

> **Work in progress.** The stacks are being rebuilt after fixes to the stacking pipeline (folders with `_rs` in the name are the rebuilt ones); results from older stacks are provisional.

In [ ]:
# Cell 1: Setup
import sys
import os
import importlib
sys.path.append('..')

import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from IPython.display import Image, display

from config import OUTPUT_DIR, LUNA_ROOT
import lake_volume as lv
importlib.reload(lv)  # re-run this cell to pick up changes in lake_volume.py
print("✓ Modules loaded")

## Parameters

The lake comes from a polygon file (GeoPackage/shapefile, one polygon per lake). `LAKE_ROWS` maps names to rows in the default candidates file; for another file, set `POLYGON_FILE` and `LAKE_ROW` directly.

`REFERENCE` chooses the coregistration when a tile has both: `'cs2'` (CryoSat-2, works inland) or `'mosaic'` (needs exposed bedrock nearby). `SUBFOLDER` forces one specific stack folder (see the list printed in step 1).

In [ ]:
# Cell 2: Parameters
POLYGON_FILE = os.path.join(LUNA_ROOT, "moralpom", "globe", "data", "ArcticDEM",
                            "candidates_polygons_02sept26.gpkg")
LAKE_ROWS = {  # row in POLYGON_FILE for each named lake
    "Flade Isblink": 1, "Sermeq SW": 9, "Sermeq Centre": 10, "Sermeq NE": 11,
    "Inuppaat Quuaat 01 (Edge)": 13, "Upernavik W": 19, "Upernavik E": 20,
    "Candidate #8": 26, "Candidate #7": 27, "Candidate #6": 28, "Sermeq (Jade)": 29,
    "Candidate #9": 30, "Candidate #4": 31,
}
LAKE_NAME = "Candidate #8"           # <-- CHANGE THIS
LAKE_ROW = LAKE_ROWS.get(LAKE_NAME)  # or a row number, for another file

REFERENCE = "cs2"      # 'cs2', 'mosaic', or None (only if the tile has one of them)
SUBFOLDER = None       # None = automatic; or a folder name from the list in step 1

RING_GAP_FRAC = 0.5    # background ring starts 0.5 r outside the lake (r = equivalent radius)...
RING_WIDTH_FRAC = 1.0  # ...and is 1 r wide
EPOCH_DAYS = 30        # strips within 30 days count as one observation (epoch)
RUN_OUTLINES = True    # step 4 (outline and events); takes a few minutes per lake

out_dir = os.path.join(OUTPUT_DIR, "volume",
                       f"notebook_{LAKE_NAME.replace(' ', '_').replace('#', '')}_{datetime.now():%Y%m%d_%H%M}")
os.makedirs(out_dir, exist_ok=True)
print(f"Lake: {LAKE_NAME} (row {LAKE_ROW})")
print(f"Outputs: {out_dir}")

## 1 · Lake polygon and stack

In [ ]:
# Cell 3: Load the lake and choose the stack
lakes = gpd.read_file(POLYGON_FILE)
lake_gdf = lakes.iloc[[LAKE_ROW]].to_crs("EPSG:3413")
if "subtile" in lake_gdf.columns:
    tile_id = lake_gdf["subtile"].iloc[0][:9]
else:  # tile containing the polygon's centroid, from the mosaic index
    index = gpd.read_file(lv.TILE_INDEX_PATH).to_crs("EPSG:3413")
    tile_id = index[index.contains(lake_gdf.geometry.centroid.iloc[0])]["tile"].iloc[0]
lon, lat = lv.get_site_lonlat_from_polygon(lake_gdf)
area_m2 = float(lake_gdf.geometry.area.iloc[0])
print(f"{LAKE_NAME}: tile {tile_id}, centre ({lon:.3f}°E, {lat:.3f}°N), "
      f"area {area_m2 / 1e6:.2f} km²")

print("\nStacks for this tile:")
for name in lv.list_tile_zarr_subfolders(tile_id):
    print("  ", name)
subfolder = SUBFOLDER or lv.select_preferred_subfolder(tile_id, reference=REFERENCE)
strip_index = lv.build_strip_index(tile_id, subfolder=subfolder)
dates = sorted(strip_index)
print(f"\nUsing: {subfolder}")
print(f"{sum(map(len, strip_index.values()))} strips, {dates[0]:%Y-%m-%d} to {dates[-1]:%Y-%m-%d}")

## 2 · Time series: lake surface minus background

For every strip: the mean elevation over the lake polygon minus the mean over a ring around it (starting `RING_GAP_FRAC` × r outside the lake, `RING_WIDTH_FRAC` × r wide). Subtracting the ring removes each strip's own vertical offset and regional thinning, leaving what the lake surface does. No outliers are removed, because a real drainage can look like one.

In [ ]:
# Cell 4: Time series
ts_df, diag = lv.build_buffer_corrected_timeseries(
    strip_index, lake_gdf, tile_id, return_diagnostics=True,
    ring_gap_frac=RING_GAP_FRAC, ring_width_frac=RING_WIDTH_FRAC)
ts_df.to_csv(os.path.join(out_dir, "timeseries.csv"), index=False)
print(diag["status"].value_counts().to_string())
print(f"\n{len(ts_df)} strips with data over the lake and the ring")

ts_png = lv.plot_polygon_timeseries(
    ts_df, lake_name=LAKE_NAME, remove_outliers=False, site_coords_4326=(lon, lat),
    output_path=os.path.join(out_dir, "timeseries.png"))
bg_png = lv.plot_lake_vs_background(
    ts_df, lake_name=LAKE_NAME, output_path=os.path.join(out_dir, "lake_vs_background.png"))
for png in (ts_png, bg_png):
    if png:
        display(Image(png))

## 3 · Ice thickness and d/H

BedMachine Greenland v6 thickness averaged over the lake (rounded to 10 m), with the bed error. d is the diameter of a circle with the lake's area. Lakes much wider than the ice is thick (d/H ≫ 1) show their basal changes almost fully at the surface; thin ice (tens of metres) suggests the lake isn't subglacial at all.

In [ ]:
# Cell 5: Ice thickness and d/H
ice = lv.lake_ice_thickness(lake_gdf)
d_m = 2 * np.sqrt(area_m2 / np.pi)
H = ice["ice_thickness_m"]
print(f"Ice thickness H: {H:.0f} m (centroid {ice['ice_thickness_centroid_m']:.0f} m, "
      f"bed error ± {ice['bed_error_m']:.0f} m)")
print(f"Equivalent diameter d: {d_m:.0f} m  →  d/H = {d_m / H:.1f}" if H and np.isfinite(H)
      else f"Equivalent diameter d: {d_m:.0f} m (no thickness)")

## 4 · Outline and events (optional, slower)

1. Crops every strip to a search area around the lake (1.5 r beyond it) and maps the standard deviation of each pixel through time. The **SD outline** is where that variability exceeds 3 × the background ring's.
2. Groups strips into observations (`EPOCH_DAYS`), keeping strips that cover at least half the lake. Each observation is compared with each pixel's last earlier one. An **event** is a change above 3 × the ring's noise; its volume is summed over the lake polygon.

In [ ]:
# Cell 6: SD outline and epoch differences
if RUN_OUTLINES:
    r = np.sqrt(area_m2 / np.pi)
    search_gdf = gpd.GeoDataFrame(geometry=lake_gdf.geometry.buffer(1.5 * r), crs=lake_gdf.crs)
    min_piece_m2 = 0.10 * area_m2  # ignore outline pieces smaller than 10% of the lake

    result = lv.build_persistence_mask(
        strip_index, search_gdf, tile_id, pixel_area=4.0, lake_gdf=lake_gdf,
        periods=lv.sliding_windows_by_count(strip_index, strips_per_window=15, step_strips=7),
        min_persistence_frac=0.5, keep_strip_cache=True)
    sd_gdf, sd_thr, ring_gdf, _ = lv.sd_outline(
        result, lake_gdf, min_area_m2=min_piece_m2,
        ring_gap_frac=RING_GAP_FRAC, ring_width_frac=RING_WIDTH_FRAC)
    print(f"SD outline: threshold {sd_thr:.2f} m, area {sd_gdf.area.sum() / 1e6:.2f} km² "
          f"(lake polygon {area_m2 / 1e6:.2f} km²)")

    epoch_res = lv.build_epoch_differences(
        result["strip_cache"], strip_index, lake_gdf, result["ref_coords"], search_gdf.crs,
        epoch_days=EPOCH_DAYS, ring_gap_frac=RING_GAP_FRAC, ring_width_frac=RING_WIDTH_FRAC,
        min_piece_m2=min_piece_m2, cleaning=result["cleaning"])
    vols = lv.contour_volumes(epoch_res, lake_gdf, result["ref_coords"])
    events = vols.merge(epoch_res["intervals_df"][["interval_index", "threshold_m"]],
                        on="interval_index")
    events = events[events["mean_dz_m"].abs() > events["threshold_m"]]
    events.to_csv(os.path.join(out_dir, "events.csv"), index=False)
    print(f"\n{len(epoch_res['epochs_df'])} observations; events over the lake polygon:")
    if len(events):
        print(events.assign(volume_Mm3=events["volume_m3"] / 1e6)[
            ["from_epoch_end", "to_epoch_start", "mean_dz_m", "threshold_m", "volume_Mm3"]
        ].round(2).to_string(index=False))
    else:
        print("  none above the threshold")

    diag_png, _ = lv.plot_persistence_diagnostics(
        result, lake_gdf, search_gdf, lake_name=LAKE_NAME, output_dir=out_dir,
        sd_outline_gdf=sd_gdf, sd_threshold_m=sd_thr, ring_gdf=ring_gdf)
    ep_png = lv.plot_epoch_differences(epoch_res, lake_gdf, search_gdf, result["ref_coords"],
                                       lake_name=LAKE_NAME, output_dir=out_dir)
    result["strip_cache"] = None  # free the cropped strips
    for png in (diag_png, ep_png):
        if png:
            display(Image(png))

## Outputs

In [ ]:
# Cell 7: Files written
for f in sorted(os.listdir(out_dir)):
    print(f)